In [1]:
# Useful imports
import os
from pathlib import Path
import tempfile
import hydra
import sys

### User Configuration Section

In [2]:
RESULT_ROOT = Path("/data/saba/parnia/Project/exp/exp/simulation/closed_loop_nonreactive_agents/diffusion_planner")
NUBOARD_FILES = sorted((p for p in RESULT_ROOT.rglob("*.nuboard") if p.stat().st_size > 0), key=lambda p: p.stat().st_mtime)
if not NUBOARD_FILES:
    raise FileNotFoundError(f"No nuBoard results found below {RESULT_ROOT}. Run sim_diffusion_planner_runner.sh first.")
print(f"Opening all {len(NUBOARD_FILES)} completed result set(s):")
for nuboard_file in NUBOARD_FILES:
    print(f"  {nuboard_file}")
env_variables = {
    "NUPLAN_DEVKIT_ROOT": "/data/saba/parnia/nuplan-devkit",
    "NUPLAN_DATA_ROOT": "/data/saba/parnia/Project/data/data/cache/mini",
    "NUPLAN_MAPS_ROOT": "/data/saba/parnia/nuplan_maps",
    "NUPLAN_EXP_ROOT": "/data/saba/parnia/Project/exp",
    "NUPLAN_SIMULATION_ALLOW_ANY_BUILDER":"1"
}

for k, v in env_variables.items():
    os.environ[k] = v

# Location of path with all nuBoard configs
CONFIG_PATH = '/data/saba/parnia/nuplan-devkit/nuplan/planning/script/config/nuboard'

Opening all 1 completed result set(s):
  /data/saba/parnia/Project/exp/exp/simulation/closed_loop_nonreactive_agents/diffusion_planner/mini/diffusion_planner_release/model_2026-08-08-18-35-28/nuboard_1786214141.nuboard


In [6]:
CONFIG_NAME = 'default_nuboard'

# Initialize configuration management system
hydra.core.global_hydra.GlobalHydra.instance().clear()  # reinitialize hydra if already initialized
hydra.initialize_config_dir(config_dir=CONFIG_PATH)

ml_planner_simulation_folder = [str(path) for path in NUBOARD_FILES]

# Compose the configuration
cfg = hydra.compose(config_name=CONFIG_NAME, overrides=[
    'scenario_builder=nuplan_mini',
    'scenario_builder.data_root=/data/saba/parnia/Project/data/data/cache/mini',
    f'simulation_path={ml_planner_simulation_folder}',  # nuboard file path(s), if left empty the user can open the file inside nuBoard
    'hydra.searchpath=[pkg://diffusion_planner.config.scenario_filter, pkg://diffusion_planner.config, pkg://nuplan.planning.script.config.common, pkg://nuplan.planning.script.experiments]',
    'port_number=8002'
])

In [7]:
from nuplan.planning.script.run_nuboard import main as main_nuboard

# Run nuBoard
main_nuboard(cfg)

INFO:nuplan.planning.script.builders.scenario_building_builder:Building AbstractScenarioBuilder...
INFO:nuplan.planning.script.builders.scenario_building_builder:Building AbstractScenarioBuilder...DONE!
INFO:nuplan.planning.nuboard.nuboard:Opening Bokeh application on http://localhost:8002/
INFO:nuplan.planning.nuboard.nuboard:Async rendering is set to: True
INFO:bokeh.server.server:Starting Bokeh server version 2.4.3 (running on Tornado 6.5.8)
INFO:bokeh.server.tornado:User authentication hooks NOT provided (default user enabled)
(node:177872) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
INFO:nuplan.planning.nuboard.base.simulation_tile:Minimum frame time=0.017 s
INFO:nuplan.planning.nuboard.tabs.scenario_tab:Rending scenario plot takes 0.0013 se

KeyboardInterrupt: 